# Supervised Classification Modeling & Tuning

**Goal:** Compare at least four classification algorithms, tune hyperparameters with `GridSearchCV` and stratified cross-validation, evaluate classification performance, select a champion model, and save it as a serialized `.joblib` file.

## Important dataset note
No dataset was supplied with this task screenshot. To make the notebook runnable immediately, it creates a reproducible **synthetic binary-classification dataset** with numeric features and missing values. The results are educational only and do not represent a real-world problem. If the internship expects a specific real dataset, replace the dataset-generation cell with that approved dataset and document its source and license.

## Workflow
1. Create/load data and inspect quality.
2. Split train/test before fitting preprocessing.
3. Build a leak-free preprocessing pipeline.
4. Tune four different model families using `GridSearchCV` and stratified K-fold cross-validation.
5. Compare precision, recall, F1, ROC-AUC and confusion matrices.
6. Plot ROC curves and select the champion using mean cross-validated ROC-AUC.
7. Save the champion pipeline to `champion_model.joblib`.


In [ ]:
# Install missing packages from a notebook cell only if needed:
# %pip install pandas numpy matplotlib scikit-learn joblib

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, classification_report
)
import joblib

RANDOM_STATE = 42


## 1. Create a reproducible tabular dataset

This cell uses a synthetic dataset so that the notebook runs without external downloads. The data contains numerical features and an imbalanced binary target. We add a few missing values to demonstrate imputation.


In [ ]:
X_array, y_array = make_classification(
    n_samples=1500,
    n_features=12,
    n_informative=7,
    n_redundant=3,
    n_repeated=0,
    n_classes=2,
    weights=[0.68, 0.32],
    class_sep=0.9,
    random_state=RANDOM_STATE
)

feature_names = [f"feature_{i:02d}" for i in range(1, 13)]
df = pd.DataFrame(X_array, columns=feature_names)
df["target"] = y_array

rng = np.random.default_rng(RANDOM_STATE)
for col in ["feature_02", "feature_07", "feature_11"]:
    missing_mask = rng.random(len(df)) < 0.04
    df.loc[missing_mask, col] = np.nan

display(df.head())
print("Shape:", df.shape)
print("Missing values by column:")
display(df.isna().sum().to_frame("missing_count"))
print("Target distribution:")
display(df["target"].value_counts().rename_axis("target").to_frame("count"))


## 2. Split the data before preprocessing

The held-out test set is separated before imputers or scalers are fitted. All learned preprocessing is placed inside the pipeline and tuned using training folds only, reducing the risk of data leakage.


In [ ]:
TARGET = "target"
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)}")
print("Numeric features:", numeric_features)


## 3. Define preprocessing and four model families

All four candidates share the same preprocessing pipeline:
- Median imputation for missing numeric values.
- Standard scaling for numeric features.
- Logistic Regression.
- Random Forest.
- Extra Trees.
- Gradient Boosting.

Tree-based models do not strictly require scaling, but using the same preprocessing structure makes the comparison workflow easier to maintain. Scaling is especially useful for Logistic Regression.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features)
    ],
    remainder="drop"
)

candidate_models = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
        {
            "model__C": [0.1, 1.0, 10.0],
            "model__solver": ["lbfgs"]
        }
    ),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [None, 8],
            "model__min_samples_leaf": [1, 3]
        }
    ),
    "Extra Trees": (
        ExtraTreesClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [None, 8],
            "model__min_samples_leaf": [1, 3]
        }
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            "model__n_estimators": [100, 200],
            "model__learning_rate": [0.03, 0.1],
            "model__max_depth": [1, 2]
        }
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


## 4. Hyperparameter tuning with GridSearchCV

The search uses stratified 5-fold cross-validation and ROC-AUC as the selection metric. The test set is not used to tune parameters.


In [ ]:
searches = {}
best_estimators = {}
summary_rows = []

for name, (estimator, param_grid) in candidate_models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=False
    )
    search.fit(X_train, y_train)
    searches[name] = search
    best_estimators[name] = search.best_estimator_
    summary_rows.append({
        "model": name,
        "best_cv_roc_auc": search.best_score_,
        "best_params": search.best_params_
    })
    print(f"{name}: best CV ROC-AUC = {search.best_score_:.4f}")

cv_results = pd.DataFrame(summary_rows).sort_values("best_cv_roc_auc", ascending=False)
display(cv_results)


## 5. Evaluate all tuned models on the held-out test set

Precision, recall, F1-score, ROC-AUC, accuracy, and balanced accuracy are reported. Choose metrics according to the real costs of false positives and false negatives; do not rely on accuracy alone.


In [ ]:
evaluation_rows = []
test_predictions = {}
test_probabilities = {}

for name, model in best_estimators.items():
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    test_predictions[name] = pred
    test_probabilities[name] = prob
    evaluation_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1_score": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, prob)
    })

evaluation = pd.DataFrame(evaluation_rows).sort_values("roc_auc", ascending=False)
display(evaluation.round(4))


## 6. Plot ROC curves and confusion matrices


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, prob in test_probabilities.items():
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
ax.set_title("ROC Curves — Held-out Test Set")
plt.tight_layout()
plt.show()


In [ ]:
for name, pred in test_predictions.items():
    print(f"\\n{name}")
    print(classification_report(y_test, pred, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, values_format="d")
    plt.title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.show()


## 7. Select the champion model

The champion is selected using the highest **mean cross-validated ROC-AUC** on the training data. The held-out test metrics are reported separately as a final evaluation, not used to tune the model.


In [ ]:
champion_name = cv_results.iloc[0]["model"]
champion_model = best_estimators[champion_name]

print("Champion model:", champion_name)
print("Best mean CV ROC-AUC:", round(float(cv_results.iloc[0]["best_cv_roc_auc"]), 4))
print("Held-out test ROC-AUC:", round(roc_auc_score(y_test, champion_model.predict_proba(X_test)[:, 1]), 4))
print("Selected parameters:", searches[champion_name].best_params_)


## 8. Serialize the champion pipeline

The saved artifact includes preprocessing and the trained estimator together. Keep the Python and Scikit-Learn versions recorded for reproducibility.


In [ ]:
artifact_path = Path("champion_model.joblib")
joblib.dump(champion_model, artifact_path)

print("Saved model to:", artifact_path.resolve())
print("scikit-learn version:", __import__("sklearn").__version__)


## 9. Load and use the saved model

Run this after the serialization cell. The example uses one row from the held-out test set.


In [ ]:
loaded_model = joblib.load("champion_model.joblib")
example_row = X_test.iloc[[0]]
print("Predicted class:", int(loaded_model.predict(example_row)[0]))
print("Predicted probabilities [class 0, class 1]:", loaded_model.predict_proba(example_row)[0])


## 10. Conclusion and responsible-use checklist

- Four distinct model families were tuned using stratified 5-fold cross-validation.
- A held-out test set was used for the final metrics and ROC curves.
- The champion was selected by training-fold CV ROC-AUC and serialized as `champion_model.joblib`.
- The dataset is synthetic. Reported scores are only a coding demonstration and must not be presented as real-world performance.
- For real work, document dataset source and permission, define the target and prediction window, inspect class imbalance and leakage, and choose thresholds based on error costs.
- To reproduce results, keep the dataset-generation cell and random seed unchanged. If using a real dataset, document the source/license and keep the test set untouched until final evaluation.
